In [3]:
import pandas as pd
import numpy as np

print("📥 Loading trial dataset...")

df = pd.read_csv("../output/training_data_v6_no0min_trial.csv", encoding="utf-8-sig")
print(f"   → Loaded {len(df):,} rows and {len(df.columns)} columns")

# Sort correctly
df = df.sort_values(["Player UUID", "season", "Gameweek"]).reset_index(drop=True)

# Create target (NextGW total points)
df["Target_NextGW"] = df.groupby(["Player UUID", "season"])["Total Points"].shift(-1)

# Remove last GW of every season/player
before = len(df)
df = df.dropna(subset=["Target_NextGW"]).reset_index(drop=True)
after = len(df)

print(f"🧹 Removed rows with no next-GW target: {before - after}")

df["Target_NextGW"] = df["Target_NextGW"].astype(float)

print("✔ Phase 1 complete.")


📥 Loading trial dataset...
   → Loaded 25,735 rows and 52 columns
🧹 Removed rows with no next-GW target: 2314
✔ Phase 1 complete.


In [4]:

print("📐 Building last-5-observations validation split per player (global across seasons)...")

# Όλα αρχικά Train
df["Split"] = "Train"

# Για κάθε παίκτη, παίρνουμε ΟΛΕΣ τις παρατηρήσεις του (όλες τις seasons),
# ήδη ταξινομημένες κατά season + Gameweek,
# και κάνουμε τις ΤΕΛΕΥΤΑΙΕΣ 5 Validation.
for uuid, g in df.groupby("Player UUID", sort=False):
    # g είναι ήδη sorted λόγω του sort_values στην αρχή
    last_5_idx = g.tail(5).index          # τελευταίες 5 γραμμές του παίκτη
    df.loc[last_5_idx, "Split"] = "Validation"

print(df["Split"].value_counts())
print("✔ Phase 2 complete.")



📐 Building last-5-observations validation split per player (global across seasons)...
Split
Train         19942
Validation     3479
Name: count, dtype: int64
✔ Phase 2 complete.


In [5]:
# %%
# ================================================================
# DEBUGGING — VERIFY LAST-5 VALIDATION SPLIT PER PLAYER
# ================================================================

print("🔍 DEBUG CHECK — Showing last 5 rows per player (season, GW, Split):\n")

debug_count = 0

for uuid, g in df.groupby("Player UUID", sort=False):

    # show only first 5 players to avoid giant output
    if debug_count < 5:
        print(f"PLAYER UUID: {uuid}")
        print(g.tail(5)[["season", "Gameweek", "Total Points", "Target_NextGW", "Split"]])
        print("-" * 70)
        debug_count += 1

print("\n⚠️ DEBUG COMPLETE — Review above output to confirm correct split.")


🔍 DEBUG CHECK — Showing last 5 rows per player (season, GW, Split):

PLAYER UUID: 00d29d0a-98c1-4730-9e17-1d278cb660ae
     season  Gameweek  Total Points  Target_NextGW       Split
14  2023-24        32             1            1.0  Validation
15  2023-24        34             1            1.0  Validation
16  2023-24        35             1            0.0  Validation
17  2023-24        36             0            2.0  Validation
18  2023-24        37             2            2.0  Validation
----------------------------------------------------------------------
PLAYER UUID: 00e5b1e6-a33d-4079-aa0e-b31b4c7d2b05
     season  Gameweek  Total Points  Target_NextGW       Split
24  2024-25        33             6            1.0  Validation
25  2024-25        34             1            3.0  Validation
26  2025-26         3             3            0.0  Validation
27  2025-26         4             0            0.0  Validation
28  2025-26         5             0            1.0  Validation
----

In [6]:
uuid = "b94be940-2996-467f-be3e-4e400fa9dd3c"   # Salah
print(df[df["Player UUID"] == uuid][["season","Gameweek","Split"]])


Empty DataFrame
Columns: [season, Gameweek, Split]
Index: []


In [7]:
print("🔧 Preparing features and target...")

df_model = df.copy()

# Drop leakage columns
cols_to_drop = [
    "Player Name", "Web Name",
    "Player Team Name", "Opponent Name",
    "season", "Gameweek"
]
df_model = df_model.drop(columns=cols_to_drop, errors="ignore")

# Target
y = df_model["Target_NextGW"].astype(float)

# Features
X = df_model.drop(columns=["Target_NextGW"], errors="ignore")

# Drop duplicate-position column if exists
if "element_type" in X.columns:
    X = X.drop(columns=["element_type"])

# Fill missing positions if any
if "Position" in X.columns:
    X["Position"] = X["Position"].fillna("Unknown")
    X = pd.get_dummies(X, columns=["Position"], drop_first=True)

print("📦 Feature matrix shape:", X.shape)
print("✔ Phase 3 complete.")


🔧 Preparing features and target...
📦 Feature matrix shape: (23421, 48)
✔ Phase 3 complete.


In [8]:
from sklearn.preprocessing import StandardScaler

print("📊 Creating train/validation masks...")

train_mask = df["Split"] == "Train"
val_mask   = df["Split"] == "Validation"

# Align X and y after clean preparation
X = X.reset_index(drop=True)
y = y.reset_index(drop=True)
df = df.reset_index(drop=True)

# Train / Val split
X_train = X.loc[train_mask].reset_index(drop=True)
X_val   = X.loc[val_mask].reset_index(drop=True)

y_train = y.loc[train_mask].reset_index(drop=True)
y_val   = y.loc[val_mask].reset_index(drop=True)

print("   → Train rows:", len(X_train))
print("   → Val rows:  ", len(X_val))

# -------------------------------------------------------
# SAFETY CHECK: Only numeric columns allowed
# -------------------------------------------------------

non_numeric = [c for c in X_train.columns if not np.issubdtype(X_train[c].dtype, np.number)]
if non_numeric:
    print("⚠ Removing non-numeric columns:", non_numeric)
    X_train = X_train.drop(columns=non_numeric)
    X_val   = X_val.drop(columns=non_numeric)
    X       = X.drop(columns=non_numeric)

# -------------------------------------------------------
# SCALING (fit only on train)
# -------------------------------------------------------
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_all_scaled   = scaler.transform(X)

print("✔ Phase 4 complete.")


📊 Creating train/validation masks...
   → Train rows: 19942
   → Val rows:   3479
⚠ Removing non-numeric columns: ['Player UUID', 'Is Home', 'Split', 'Position_FWD', 'Position_GK', 'Position_MID']
✔ Phase 4 complete.


In [9]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("🤖 Training Linear Regression model...")

model = LinearRegression()
model.fit(X_train_scaled, y_train)

y_train_pred = model.predict(X_train_scaled)
y_val_pred   = model.predict(X_val_scaled)
y_all_pred   = model.predict(X_all_scaled)

# Metrics
train_mae = mean_absolute_error(y_train, y_train_pred)
val_mae   = mean_absolute_error(y_val, y_val_pred)
val_rmse  = mean_squared_error(y_val, y_val_pred) ** 0.5

print("\n📈 PERFORMANCE:")
print(f"   • MAE Train:      {train_mae:.4f}")
print(f"   • MAE Validation: {val_mae:.4f}")
print(f"   • RMSE Validation:{val_rmse:.4f}")

print("✔ Phase 5 complete.")


🤖 Training Linear Regression model...

📈 PERFORMANCE:
   • MAE Train:      2.1105
   • MAE Validation: 1.7527
   • RMSE Validation:2.5063
✔ Phase 5 complete.


In [10]:
from pathlib import Path

OUT_DIR = Path("../output/linear_regression_model")
OUT_DIR.mkdir(parents=True, exist_ok=True)


In [11]:
# ================================================================
# PHASE 6 — Export Linear Regression Coefficients
# ================================================================
print("📤 Exporting Linear Regression coefficients...")

# Build coefficients DataFrame
coeff_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": model.coef_
})

# 👉 ΑΦΑΙΡΟΥΜΕ ΕΝΤΕΛΩΣ ΤΗ ΣΤΗΛΗ AbsCoefficient
# (ΔΕΝ τη δημιουργούμε, ΔΕΝ ταξινομούμε με βάση αυτήν)

# Προαιρετικά: ταξινόμηση απλά με βάση το Coefficient (όχι absolute)
coeff_df = coeff_df.sort_values("Coefficient", ascending=False)

# Add intercept as a separate row
intercept_row = pd.DataFrame([{
    "Feature": "Intercept",
    "Coefficient": model.intercept_
}])

coeff_df = pd.concat([intercept_row, coeff_df], ignore_index=True)

# Save to output folder
COEFF_PATH = OUT_DIR / "coefficients.csv"
coeff_df.to_csv(COEFF_PATH, index=False, encoding="utf-8-sig")

print(f"✔ Coefficients exported to {COEFF_PATH}")
print(coeff_df.head(15))


📤 Exporting Linear Regression coefficients...
✔ Coefficients exported to ..\output\linear_regression_model\coefficients.csv
                            Feature  Coefficient
0                         Intercept     2.869271
1   Team_Points_Contribution_Causal    10.880328
2       Team_Points_Contribution_GW     7.624118
3                     Avg_Threat_L5     2.362966
4                 Avg_Creativity_L5     2.011809
5                  Avg_Influence_L5     1.921548
6                  Avg_ICT Index_L3     0.883198
7                         ICT Index     0.846872
8              Player_Season_Points     0.323957
9                    Minutes Played     0.265973
10              Avg_Total Points_L5     0.187138
11                     Total Points     0.092411
12              Opponent Difficulty     0.086959
13        Team_Contribution_Rank_GW     0.045483
14             Team_Total_Points_GW     0.042744


In [12]:
from pathlib import Path

print("💾 Saving outputs...")

OUT_DIR = Path("../output/linear_regression_model")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PERF_DIR = Path("../output/model_performance")
MODEL_PERF_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PERF_PATH = MODEL_PERF_DIR / "Model_Performance.csv"

# Rebuild df_meta for clean output
df_out = df.copy()

df_out["Prediction_For_GW"] = df_out["Gameweek"] + 1
df_out["Total_Points_Actual"] = df_out["Target_NextGW"]
df_out["Total_Points_Predicted"] = y_all_pred.round(4)
df_out["Error"] = (df_out["Total_Points_Actual"] - df_out["Total_Points_Predicted"]).abs().round(4)

keep_cols = [
    "Player UUID", "Player Name", "Web Name",
    "season", "Prediction_For_GW", "Opponent Difficulty",
    "Total_Points_Actual", "Total_Points_Predicted",
    "Error", "Split"
]

df_final = df_out[keep_cols].copy()
df_final.rename(columns={"Split":"Train_or_Validation"}, inplace=True)

df_final.to_csv(OUT_DIR / "linear_reg_predictions.csv", index=False, encoding="utf-8-sig")

print("✔ Saved linear_reg_predictions.csv")


💾 Saving outputs...
✔ Saved linear_reg_predictions.csv


In [13]:
train_csv = df_final[df_final["Train_or_Validation"] == "Train"]
val_csv   = df_final[df_final["Train_or_Validation"] == "Validation"]

train_csv.to_csv(OUT_DIR / "linear_reg_train.csv", index=False, encoding="utf-8-sig")
val_csv.to_csv(OUT_DIR / "linear_reg_validation.csv", index=False, encoding="utf-8-sig")

# Split info
split_info = df[["Player UUID", "season", "Gameweek", "Split"]]
split_info.rename(columns={"Split": "Train_or_Validation"}, inplace=True)
split_info.to_csv(OUT_DIR / "linear_reg_split.csv", index=False, encoding="utf-8-sig")

print("✔ Saved train/validation/split files.")


✔ Saved train/validation/split files.


C:\Users\SOFI\AppData\Local\Temp\ipykernel_19732\1488105629.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  split_info.rename(columns={"Split": "Train_or_Validation"}, inplace=True)


In [15]:

# PHASE — Update Model_Performance.csv for Linear Regression (TotalPoints)

print(" Updating Model_Performance.csv ...")

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import mean_squared_error

MODEL_PERF_PATH = Path("../output/model_performance/Model_Performance.csv")


#  Compute RMSE (manual sqrt – no squared=False)

train_rmse = mean_squared_error(y_train, y_train_pred) ** 0.5
val_rmse   = mean_squared_error(y_val, y_val_pred) ** 0.5


#  Load existing Model_Performance.csv

if MODEL_PERF_PATH.exists():
    perf = pd.read_csv(MODEL_PERF_PATH)

    # Remove ONLY the previous row of this model
    perf = perf[perf["Model"] != "Linear Regression (TotalPoints)"]

    # Get baseline MAE
    baseline_row = perf[perf["Model"] == "Rolling Average (TotalPoints)"]
    if len(baseline_row) > 0:
        baseline_val_mae = baseline_row.iloc[0]["MAE_Validation"]
    else:
        baseline_val_mae = val_mae   # fallback if baseline missing
else:
    perf = pd.DataFrame()
    baseline_val_mae = val_mae


#  Compute Relative Improvement

relative_improvement = (baseline_val_mae - val_mae) / baseline_val_mae
relative_improvement = round(relative_improvement, 5)


# Build NEW row for TOTAL FEATURES Linear Regression

new_row = pd.DataFrame([{
    "Model": "Linear Regression (TotalPoints)",
    "MAE_Train": round(train_mae, 5),
    "MAE_Validation": round(val_mae, 5),
    "Relative_Improvement_vs_Baseline": relative_improvement,
    "RMSE_Train": round(train_rmse, 5),
    "RMSE_Validation": round(val_rmse, 5)
}])


#  Append + Save CLEAN table

perf = pd.concat([perf, new_row], ignore_index=True)
perf.to_csv(MODEL_PERF_PATH, index=False, encoding="utf-8-sig")

print("✔ Model_Performance updated successfully for Linear Regression (TotalPoints)")


 Updating Model_Performance.csv ...
✔ Model_Performance updated successfully for Linear Regression (TotalPoints)
